# PATSTAT  first queries - lesson 1
This notebook is an introduction to the PATSTAT library in TIP. The European Patent Office's (EPO) PATSTAT (EPO's Worldwide Patent Statistical Database) is a comprehensive resource designed for advanced statistical analysis of patent data. Launched in 2006, PATSTAT consolidates data from patent offices worldwide, facilitating in-depth research and trend analysis in the field of intellectual property. It includes detailed information on patent applications, applicants, inventors, and legal events, enabling users to perform complex queries and extract valuable insights into patenting activities and innovation trends. PATSTAT is widely used by policymakers, researchers, and businesses to inform decision-making and strategy development in technology and innovation sectors.

## The PATSTAT data structure and key tables

PATSTAT is organised into a relational database structure. This design facilitates complex statistical analysis and research, structured across multiple interrelated tables, each containing specific categories of data. The data structure of PATSTAT is rather complex, with two databases containing a multitude of interrelated tables. Below you have a list of some of the main tables that we will be using in this course. 

### The two PATSTAT databases

PATSTAT in TIP has access to two primary databases.

1. **PATSTAT Global**: This database is a comprehensive collection of worldwide patent data, ideal for broad statistical analysis and research. It includes detailed information on patent applications, filings, classifications, and the entities involved. See the full documentation [here](https://link.epo.org/web/searching-for-patents/business/patstat/data-catalog-patsat-global-en.pdf).

3. **PATSTAT EP Register**: This database specifically focuses on the European Patent Register, providing detailed statistical analysis of EP applications and their status. It includes data on the legal status of patents, procedural details, and administrative events. See the full documentation [here](https://link.epo.org/web/searching-for-patents/business/patstat/data-catalog-patsat-ep-register-en.pdf).

These databases support various joins and relationships between tables through primary and foreign keys, enabling users to derive detailed insights and conduct in-depth patent analysis efficiently.



### Key tables in PATSTAT

- **tls201_appln**: contains basic information about patent applications, including application numbers, filing dates, and types of patents.
- **tls206_person**: stores details about individuals and entities involved in patent applications, such as inventors and applicants, including their names and country codes.
- **tls207_pers_appln**: acts as a link between applications and individuals, indicating which persons are associated with which applications.
- **tls209_appln_ipc**: lists the International Patent Classification (IPC) codes assigned to applications, crucial for identifying the technical field of the inventions.
- **tls224_appln_cpc**: contains Cooperative Patent Classification (CPC) codes, another classification system used to categorise patents.

This relational structure allows users to perform detailed queries and analyses. For instance, an application in the **tls201_appln** table can be linked to its inventors in the **tls207_pers_appln** and **tls206_person** tables, and to its technical classifications in the **tls209_appln_ipc** and **tls224_appln_cpc** tables. 

## Our first query with Object-Relational Mapping
PATSTAT in TIP allows you to access the PATSTAT databases with Object-Relational Mapping (ORM).  In simple terms, ORM allows you to interact with a database using Python objects instead of writing raw SQL queries. 

By using ORM, we abstract the underlying SQL queries, making the code cleaner, more maintainable, and easier to understand. This is particularly advantageous for complex queries involving multiple joins and filters, which are common when working with patent databases.

### SQLAlchemy

PATSTAT client contains an implementation of SQLAlchemy, a well known Object-Relational Mapping (ORM) library for Python. 

It is recommended that users familiarise themselves with SQLAlchemy. For more information, refer to the [official SQLAlchemy documentation](https://www.sqlalchemy.org).


### Step 1: setup PATSTAT client
To begin, you need to initialise the PATSTAT client. This client will be used to interact with the PATSTAT database.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient()

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



### Step 2: initialise ORM access
The TIP Python library includes an `orm()` method that gives structured (ORM-based) access to the PATSTAT database.

In [2]:
# Access ORM
db = patstat.orm()

print (db)

### Step 3: import necessary models
You need to import the necessary tables (models) that you want to query. For this example, we will use just one table: `TLS201_APPLN`. 

In [3]:
# Importing tables as models
from epo.tipdata.patstat.database.models import TLS201_APPLN

### Step 4: launch the query
For our first query we want to get all the EP applications that were filed in 2010 and have been granted. 

We are going to select specific fields `appln_id`, `appln_auth`, `appln_nr`, `appln_kind`, and `appln_filing_date` from the `TLS201_APPLN table` This is equivalent to a `SELECT` statement in SQL, and we do this with the `query()` method. 

The filter method adds conditions to the query, retrieving only records where the application filing year is 2010, the application authority is 'EP' (European Patent Office), and the application is granted (granted is 'Y'). 

In [4]:
# Define the query
q = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date
).filter(
    TLS201_APPLN.appln_filing_year == 2010,
    TLS201_APPLN.appln_auth == 'EP',
    TLS201_APPLN.granted == 'Y'
)

# Execute the query and convert the result to a DataFrame
result_df = patstat.df(q)

# Display the results
result_df.head()


,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date
0,274222610,EP,10000313,A,2010-01-14
1,274369023,EP,10000849,A,2010-01-28
2,274681480,EP,10001469,A,2010-02-12
3,274720647,EP,10001552,A,2010-02-16
4,274875659,EP,10002051,A,2010-03-01


## Querying PATSTAT with SQL
You can also pass SQL queries to the library, with the same method `sql_query` that you can use with the EP full-text data library. In the example below we query PATSTAT using SQL, passing the same query we just did above: granted EP patents filed in 2010.

In [5]:
res = patstat.sql_query("""
SELECT
    appln_id,
    appln_auth,
    appln_nr,
    appln_kind,
    appln_filing_date
FROM
    tls201_appln
WHERE
    appln_filing_year = 2010
    AND appln_auth = 'EP'
    AND granted = 'Y';

""")
# Printing the first 5 results to compare with the dataframe 
res[0:5]

[{'appln_id': 274222610,
  'appln_auth': 'EP',
  'appln_nr': '10000313',
  'appln_kind': 'A ',
  'appln_filing_date': datetime.date(2010, 1, 14)},
 {'appln_id': 274369023,
  'appln_auth': 'EP',
  'appln_nr': '10000849',
  'appln_kind': 'A ',
  'appln_filing_date': datetime.date(2010, 1, 28)},
 {'appln_id': 274681480,
  'appln_auth': 'EP',
  'appln_nr': '10001469',
  'appln_kind': 'A ',
  'appln_filing_date': datetime.date(2010, 2, 12)},
 {'appln_id': 274720647,
  'appln_auth': 'EP',
  'appln_nr': '10001552',
  'appln_kind': 'A ',
  'appln_filing_date': datetime.date(2010, 2, 16)},
 {'appln_id': 274875659,
  'appln_auth': 'EP',
  'appln_nr': '10002051',
  'appln_kind': 'A ',
  'appln_filing_date': datetime.date(2010, 3, 1)}]